# HICCUP: control vs. K&C bogus initial conditions for Typhoon Mawar (WP02 2023)

HICCUP (`external/hiccup`) turns ERA5 into an E3SM (EAM) initial-condition
file. It is a preprocessor, not a forecast model, so this notebook compares
**initial conditions only**. No E3SM runs are made.

From the ERA5 analysis at **2023-05-26 06Z**, about a day before Mawar's peak,
we build two sets of HICCUP input files:

1. **control**: ERA5 unchanged;
2. **bogus**: ERA5 with a Kwon & Cheong (2009) vortex matched to the JTWC
   best-track intensity, written by `tcinit.backends.hiccup_backend`.

HICCUP then remaps each set to the E3SM `ne30np4` grid with 80 levels, and we
look at how much of the vortex is left in the E3SM file.

**Run on:** cee-compute in `torch-venv` (see `envs/README.md`). HICCUP also
needs these command-line tools on `PATH`:

* **NCO ≥ 5.3.1**: `ncremap`, `ncks`, `ncdump`, `ncatted`. HICCUP finds them
  with `shutil.which` and checks the version of `ncks --version` when the
  data object is created.
* **TempestRemap**: `GenerateCSMesh` builds the ne30 mesh, and `ncremap -a
  fv2se_flx` runs `GenerateOverlapMesh` and `GenerateOfflineMap` internally.

If the modules don't provide both, HICCUP's README suggests a small conda
environment, which only needs to be on `PATH`, not activated as the kernel:
`conda create -n hiccup_tools -c conda-forge "nco>=5.3.1" tempest-remap`, then
`export PATH=$CONDA_PREFIX/bin:$PATH` before starting Jupyter. Section 1
checks for the tools before anything runs.

Reading ARCO ERA5 from `gs://` also needs `gcsfs` and `zarr`. Both are in
`torch-venv`. Run `proxy_on` before starting Jupyter.

## 1. Configuration

In [ ]:
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path

REPO = Path.cwd().resolve().parent  # notebook lives in <repo>/notebooks
HICCUP_ROOT = REPO / "external/hiccup"
sys.path.insert(0, str(HICCUP_ROOT))  # HICCUP isn't pip-installed in torch-venv

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/hiccup_mawar")

START_TIME = datetime(2023, 5, 26, 6)
BOX_HALF_DEG = 10.0

ARCO_ERA5 = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# E3SM target grid. ne30np4 with L80 is the default in HICCUP's template
# (template_hiccup_scripts/create_EAM_IC_from_ERA5-NOAA.py), and its topography
# file ships with HICCUP, so E3SM inputdata isn't needed.
DST_HORZ_GRID = "ne30np4"
DST_VERT_GRID = "L80"
VERT_FILE = HICCUP_ROOT / "files_vert/L80_for_E3SMv3.nc"
TOPO_FILE = HICCUP_ROOT / "test_data/USGS-topo_ne30-np4_smoothedx6t_20260403.nc"

ERA5_DIR = OUT_DIR / "era5"  # control + .tc_bogus.nc source files
GRID_MAP_DIR = OUT_DIR / "grid_map"  # SCRIP/exodus/map files, shared by both runs
RUN_DIRS = {"control": OUT_DIR / "control", "bogus": OUT_DIR / "bogus"}

for p in (BEST_TRACK_FILE, HICCUP_ROOT, VERT_FILE, TOPO_FILE):
    if not p.exists():
        raise FileNotFoundError(p)
missing = [
    exe
    for exe in ("ncremap", "ncks", "ncdump", "ncatted", "GenerateCSMesh",
                "GenerateOverlapMesh", "GenerateOfflineMap")
    if shutil.which(exe) is None
]
if missing:
    raise OSError(f"not on PATH: {missing} (need NCO >= 5.3.1 and TempestRemap)")
for d in (ERA5_DIR, GRID_MAP_DIR, *RUN_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")

In [ ]:
import contextlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from hiccup import hiccup

from tcinit import BestTrack
from tcinit.backends import hiccup_backend
from tcinit.constants import HPA_TO_PA
from tcinit.earth_util import haversine_dist
from tcinit.plotting import plot_snapshot
from tcinit.rollout_tracking import build_bogus, snapshot_from_box

hiccup.hu.check_nco_version()  # raises if NCO < 5.3.1

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
bt_pc_hpa = bt.get_central_pressure(START_TIME) / HPA_TO_PA
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, {bt_pc_hpa:.0f} hPa"
)

## 3. ERA5 input files from ARCO ERA5

HICCUP's ERA5 path reads two netCDF files per time, in the layout written by
`external/hiccup/get_hindcast_data.ERA5.py` (CDS API):

* `ERA5.atm.<YYYY-MM-DD>.<HH>.nc`: `t, q, u, v, z, o3, clwc, ciwc` on
  dims `(valid_time, pressure_level, latitude, longitude)`, 37 levels;
* `ERA5.sfc.<YYYY-MM-DD>.<HH>.nc`: `sp, skt, z` (surface geopotential),
  `stl1..4, siconc, sd`, plus others it ignores.

The grid must be the full 0.25° grid (721 × 1440), with latitude running north
to south and longitude 0–359.75. HICCUP's source SCRIP grid assumes that layout
(`ncremap -G latlon=721,1440#lat_drc=n2s#lon_typ=grn_ctr`).

We build the same files from the public ARCO ERA5 Zarr store, so no CDS account
is needed. We also add `msl, t2m, u10, v10` to the surface file. HICCUP ignores
them, but tcinit needs `msl` to find the storm and writes the vortex into all
four.

**Alternative (CDS):** run `python external/hiccup/get_hindcast_data.ERA5.py
--start-date=20230526 --start-hour=06 --output-root=<ERA5_DIR>` with a
`~/.cdsapirc` token. Its surface request has no `mean_sea_level_pressure`,
`10m_u_component_of_wind` or `10m_v_component_of_wind`, so add those
to a copy of the script. Without `msl` the storm can't be detected.

In [ ]:
TAG = f"{START_TIME:%Y-%m-%d}.{START_TIME:%H}"
ATM_FILE = ERA5_DIR / f"ERA5.atm.{TAG}.nc"
SFC_FILE = ERA5_DIR / f"ERA5.sfc.{TAG}.nc"

# HICCUP / CDS netCDF short name -> ARCO long name
ARCO_ATM = {
    "t": "temperature",
    "q": "specific_humidity",
    "z": "geopotential",
    "u": "u_component_of_wind",
    "v": "v_component_of_wind",
    "o3": "ozone_mass_mixing_ratio",
    "clwc": "specific_cloud_liquid_water_content",
    "ciwc": "specific_cloud_ice_water_content",
}
ARCO_SFC = {
    "sp": "surface_pressure",
    "skt": "skin_temperature",
    "z": "geopotential_at_surface",
    "stl1": "soil_temperature_level_1",
    "stl2": "soil_temperature_level_2",
    "stl3": "soil_temperature_level_3",
    "stl4": "soil_temperature_level_4",
    "siconc": "sea_ice_cover",
    "sd": "snow_depth",
    # not used by HICCUP; needed by tcinit
    "msl": "mean_sea_level_pressure",
    "t2m": "2m_temperature",
    "u10": "10m_u_component_of_wind",
    "v10": "10m_v_component_of_wind",
}


def arco_to_cds(arco, mapping, t):
    # One time slice -> CDS-style Dataset: valid_time (len 1), pressure_level
    # descending 1000 -> 1 hPa, latitude N -> S, float32.
    out = {}
    for short, long in mapping.items():
        da = arco[long]
        if "time" in da.dims:
            da = da.sel(time=np.datetime64(t))
        else:  # static field (geopotential_at_surface)
            da = da.drop_vars("time", errors="ignore")
        da = da.expand_dims(time=[np.datetime64(t, "ns")])
        out[short] = da.astype("float32").load()
    ds = xr.Dataset(out).rename(time="valid_time")
    if "level" in ds.dims:
        ds = ds.rename(level="pressure_level").sortby("pressure_level", ascending=False)
    ds = ds.sortby("latitude", ascending=False)
    ds.attrs = {"source": f"ARCO ERA5 ({ARCO_ERA5}), {t:%Y-%m-%d %HZ}"}
    return ds


def write_cds_nc(ds, path):
    # A finite _FillValue keeps HICCUP from making a `.modified.nc` copy
    # (it rewrites NaN fill values with ncatted). write_back keeps it.
    enc = {v: {"_FillValue": 1.0e36} for v in ds.data_vars}
    enc.update({c: {"_FillValue": None} for c in ("latitude", "longitude")})
    enc["valid_time"] = {"units": "seconds since 1970-01-01", "dtype": "int64"}
    ds.to_netcdf(path, encoding=enc)


if not (ATM_FILE.exists() and SFC_FILE.exists()):
    arco = xr.open_zarr(ARCO_ERA5, chunks=None, storage_options=dict(token="anon"))
    absent = sorted((set(ARCO_ATM.values()) | set(ARCO_SFC.values())) - set(arco.data_vars))
    if absent:
        raise KeyError(f"ARCO store lacks {absent}")  # CHECK ON HPC: ARCO long names
    write_cds_nc(arco_to_cds(arco, ARCO_ATM, START_TIME), ATM_FILE)
    write_cds_nc(arco_to_cds(arco, ARCO_SFC, START_TIME), SFC_FILE)

with xr.open_dataset(ATM_FILE) as a, xr.open_dataset(SFC_FILE) as s:
    print(ATM_FILE.name, dict(a.sizes), list(a.data_vars))
    print(SFC_FILE.name, dict(s.sizes), list(s.data_vars))
    assert a.sizes["latitude"] == 721 and a.sizes["longitude"] == 1440
    assert float(a.latitude[0]) > float(a.latitude[-1])  # north to south

## 4. Control source state

`extract_snapshot_dataset` reads both files and maps CDS names (`t2m, u10,
v10, pressure_level, valid_time`) to tcinit names. It also includes `sp`,
because HICCUP takes E3SM `PS` from `sp`. Both files have a variable called
`z`. Only the pressure-level `z` goes into the box, so file order doesn't
matter.

In [ ]:
CTRL_FILES = [str(ATM_FILE), str(SFC_FILE)]  # atm first, as in HICCUP's template


def extract_box(files, center):
    return hiccup_backend.extract_snapshot_dataset(
        files, box_center=center, box_size=BOX_HALF_DEG
    )


box_ctrl = extract_box(CTRL_FILES, bt_center)
assert box_ctrl["z"].ndim == 3 and "sp" in box_ctrl  # pressure-level z; sp for HICCUP
snap_ctrl = snapshot_from_box(box_ctrl, valid_time=START_TIME, search_center=bt_center)
print(
    f"ERA5 storm at ({snap_ctrl.loc[0]:.2f}, {snap_ctrl.loc[1]:.2f}), "
    f"MSLP min {snap_ctrl.mslp_min / HPA_TO_PA:.1f} hPa (best track {bt_pc_hpa:.0f} hPa)"
)
plot_snapshot(
    snap_ctrl,
    box_ctrl,
    title=f"Mawar, ERA5 {START_TIME:%Y-%m-%d %HZ} (control source)",
)
plt.show()

## 5. Bogus source files

`build_bogus` finds the storm, builds the K&C vortex from the best-track
intensity, and applies it with `hiccup_backend.apply`. `write_back` then copies
each source file to a `.tc_bogus.nc` sibling with the box replaced. It keeps
each variable's source `_FillValue` and storage encoding, so HICCUP doesn't need
to make `.modified.nc` copies. Variables the vortex doesn't change (`o3, clwc,
ciwc, skt`, surface `z`, soil, ice) are copied as they are.

**What HICCUP actually reads.** HICCUP's EAM variable map
(`hiccup_data_subclass_ERA5.py:74-93`) takes `T, Q, U, V` from `t, q, u, v`,
`PS` from **`sp`**, and `TS` from **`skt`**. It doesn't read `msl`, `t2m`,
`u10`, `v10` or pressure-level `z`. So the vortex's pressure drop reaches E3SM
only through `sp`. `hiccup_backend.apply` sets `sp_bogus = sp · msl_bogus / msl`,
which keeps the control's `sp / msl` ratio (close to 1 over the ocean).
HICCUP's surface-pressure adjustment then only corrects for the topography
difference. The wind, temperature and moisture changes reach E3SM through
`u, v, t, q`.

In [ ]:
_, bv, box_bogus = build_bogus(
    box_ctrl, bt, START_TIME, search_center=bt_center, apply=hiccup_backend.apply
)
print(f"p_c  = {bv.p_c / HPA_TO_PA:.1f} hPa (best track)")
print(f"V_m  = {bv.V_m:.1f} m/s")
print(f"R_v  = {bv.R_v_km:.0f} km")

bogus_files = [
    hiccup_backend.write_back(
        f,
        box_bogus,
        box_lats=box_ctrl.latitude.values,
        box_lons=box_ctrl.longitude.values,
        overwrite=True,
    )
    for f in CTRL_FILES
]
BOGUS_ATM_FILE, BOGUS_SFC_FILE = map(Path, bogus_files)
print(BOGUS_ATM_FILE.name, BOGUS_SFC_FILE.name)

In [ ]:
BOGUS_FILES = [str(BOGUS_ATM_FILE), str(BOGUS_SFC_FILE)]
box_bogus_chk = extract_box(BOGUS_FILES, bt_center)
snap_bogus = snapshot_from_box(box_bogus_chk, valid_time=START_TIME, search_center=bt_center)
print(
    f"MSLP min: control {snap_ctrl.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus.mslp_min / HPA_TO_PA:.1f} hPa"
)
print(
    f"sp min:   control {float(box_ctrl['sp'].min()) / HPA_TO_PA:.1f} hPa -> "
    f"bogus {float(box_bogus_chk['sp'].min()) / HPA_TO_PA:.1f} hPa"
)
plot_snapshot(
    snap_bogus,
    box_bogus_chk,
    title=f"Mawar, ERA5 + K&C bogus {START_TIME:%Y-%m-%d %HZ} (bogus source)",
)
plt.show()

## 6. HICCUP: ERA5 → E3SM initial conditions

These steps follow `template_hiccup_scripts/create_EAM_IC_from_ERA5-NOAA.py`
in order, without the SST/sea-ice part:

1. `create_hiccup_data`: build the data object and check the input variables;
2. `create_src_grid_file`, `create_dst_grid_file`, `create_map_file`: the
   0.25° SCRIP grid, the ne30np4 exodus mesh, and the `fv2se_flx` map;
3. `remap_horizontal_multifile`, `rename_vars_multifile`,
   `add_time_date_variables_multifile`;
4. `surface_adjustment_multifile`: adjust PS to the E3SM topography;
5. `remap_vertical_multifile`: interpolate from pressure levels to L80
   hybrid levels;
6. `atmos_state_adjustment_multifile`: remove supersaturation, clip negative
   cloud water, convert O3 to a volume mixing ratio;
7. `combine_files` and `clean_global_attributes`.

**Why ne30np4.** It is the default grid in HICCUP's template and end-to-end
test, its topography file ships in `external/hiccup/test_data/`, and the grid
files take seconds to make, so the cost is dominated by reading the 0.25°
ERA5 files. EAM's `pg2` physics grids aren't an option here, because for
`target_model='EAM'` HICCUP only sets up the `np4` path. ne30np4 has GLL nodes
about 110 km apart, so it is also a fair test of how much of a vortex with a
radius of maximum wind of tens of km survives. For an ~25 km grid, change to
`ne120np4` and point `TOPO_FILE` at
`atm/cam/topo/USGS-gtopo30_ne120np4_16xdel2-PFC-consistentSGH.nc` from E3SM
inputdata (see `hiccup.get_default_topo_file_name`).

Both runs share one grid/map directory, since the source and target grids are
the same. Each run has its own `tmp/` and output directory. TempestRemap writes
`TempestRemap.log` to the working directory, so each run changes into its own
run directory first.

In [ ]:
def ic_file(run):
    return RUN_DIRS[run] / f"HICCUP.atm_era5.{TAG}.{DST_HORZ_GRID}.{DST_VERT_GRID}.nc"


def run_hiccup(run, input_files, make_maps):
    run_dir = RUN_DIRS[run]
    tmp_dir = run_dir / "tmp"
    tmp_dir.mkdir(exist_ok=True)  # create_hiccup_data does not create tmp_dir
    out_file = ic_file(run)
    with contextlib.chdir(run_dir):
        hd = hiccup.create_hiccup_data(
            src_data_name="ERA5",
            target_model="EAM",
            dst_horz_grid=DST_HORZ_GRID,
            dst_vert_grid=DST_VERT_GRID,
            input_file_list=list(input_files),  # atm first, as in the template
            topo_file=str(TOPO_FILE),
            output_dir=str(run_dir),
            grid_dir=str(GRID_MAP_DIR),
            map_dir=str(GRID_MAP_DIR),
            tmp_dir=str(tmp_dir),
            verbose=True,
            check_input_files=True,
        )
        file_dict = hd.get_multifile_dict()

        if make_maps or not Path(hd.map_file).exists():
            hd.create_src_grid_file()
            hd.create_dst_grid_file()
            hd.create_map_file()

        hd.remap_horizontal_multifile(file_dict)
        hd.rename_vars_multifile(file_dict=file_dict)
        hd.add_time_date_variables_multifile(file_dict=file_dict)
        hd.surface_adjustment_multifile(file_dict=file_dict)
        hd.remap_vertical_multifile(file_dict=file_dict, vert_file_name=str(VERT_FILE))
        hd.atmos_state_adjustment_multifile(file_dict=file_dict)
        hd.combine_files(file_dict=file_dict, delete_files=True, output_file_name=str(out_file))
        hd.clean_global_attributes(file_name=str(out_file))
        hd.print_timer_summary()
    return out_file

In [ ]:
IC = {
    "control": run_hiccup("control", CTRL_FILES, make_maps=True),
    "bogus": run_hiccup("bogus", BOGUS_FILES, make_maps=False),
}
for k, v in IC.items():
    print(f"{k:8s} {v}")

## 7. Compare the two E3SM initial conditions

The EAM IC is unstructured (`ncol` = 48 602 GLL nodes), so we use the `lat`/`lon`
stored in the file, falling back to `lat_d`/`lon_d` from the topography file.
EAM levels run from the top down, so `lev = -1` is the lowest model level
(about 10–20 m above the surface in L80).

In [ ]:
def open_ic(path):
    ds = xr.open_dataset(path).isel(time=0)
    if "lat" not in ds or "lon" not in ds:
        # CHECK ON HPC: assumes ncol in the IC matches ncol_d order in the topo file
        with xr.open_dataset(TOPO_FILE) as topo:
            ds = ds.assign(
                lat=("ncol", topo["lat_d"].values), lon=("ncol", topo["lon_d"].values)
            )
    return ds


ic = {k: open_ic(v) for k, v in IC.items()}
lat, lon = ic["control"]["lat"].values, ic["control"]["lon"].values % 360.0
assert np.allclose(ic["bogus"]["lat"].values, lat), "control and bogus grids differ"
print({v: ic["control"][v].dims for v in ("PS", "T", "U", "V") if v in ic["control"]})

dist_km = haversine_dist(bt_center[0], bt_center[1], lat, lon)
near = (np.abs(lat - bt_center[0]) <= BOX_HALF_DEG) & (
    np.abs(((lon - bt_center[1] + 180) % 360) - 180) <= BOX_HALF_DEG
)
print(f"{near.sum()} GLL nodes within ±{BOX_HALF_DEG}° of the best-track centre")


def ic_fields(ds):
    bot = ds.isel(lev=-1)
    return {
        "PS": ds["PS"].values / HPA_TO_PA,
        "T_bot": bot["T"].values,
        "wspd_bot": np.hypot(bot["U"].values, bot["V"].values),
    }


fld = {k: ic_fields(ds) for k, ds in ic.items()}

In [ ]:
plot_spec = {
    "PS": ("PS (hPa)", "viridis"),
    "T_bot": ("T, lowest level (K)", "inferno"),
    "wspd_bot": ("|V|, lowest level (m/s)", "magma_r"),
}
x, y = lon[near], lat[near]
fig, axes = plt.subplots(3, 3, figsize=(16, 14), sharex=True, sharey=True)
for row, (name, (label, cmap)) in enumerate(plot_spec.items()):
    c, b = fld["control"][name][near], fld["bogus"][name][near]
    levels = np.linspace(min(c.min(), b.min()), max(c.max(), b.max()), 31)
    for col, (data, ttl) in enumerate(((c, "control"), (b, "bogus"))):
        cf = axes[row, col].tricontourf(x, y, data, levels=levels, cmap=cmap)
        axes[row, col].set_title(f"{ttl}: {label}")
        fig.colorbar(cf, ax=axes[row, col], shrink=0.8)
    d = b - c
    lim = float(np.nanmax(np.abs(d))) or 1.0
    cf = axes[row, 2].tricontourf(x, y, d, levels=np.linspace(-lim, lim, 31), cmap="RdBu_r")
    axes[row, 2].set_title(f"bogus − control: {label}")
    fig.colorbar(cf, ax=axes[row, 2], shrink=0.8)
for ax in axes.flat:
    ax.plot(bt_center[1], bt_center[0], "kx", ms=10, mew=2)
    ax.scatter(x, y, s=1, c="k", alpha=0.15)  # GLL nodes
    ax.set_aspect("equal")
fig.suptitle(f"E3SM IC on {DST_HORZ_GRID} {DST_VERT_GRID}, Mawar {START_TIME:%Y-%m-%d %HZ}")
plt.tight_layout()
plt.show()

### What survives the remap

The minimum pressure and maximum wind at each stage: the 0.25° source
(`msl`, `sp`, 10 m wind) and the ne30np4 IC (`PS`, lowest-level wind), compared
with the best-track central pressure. The difference between the source and IC
rows is what the conservative `fv2se_flx` remap to ~1° GLL nodes, plus the
surface adjustment, removes from the vortex.

In [ ]:
def source_stats(files):
    with xr.open_dataset(files[1]) as s:
        s = s.isel(valid_time=0)
        box = dict(
            latitude=slice(bt_center[0] + BOX_HALF_DEG, bt_center[0] - BOX_HALF_DEG),
            longitude=slice(bt_center[1] - BOX_HALF_DEG, bt_center[1] + BOX_HALF_DEG),
        )
        s = s.sel(box)
        return {
            "min msl (hPa)": float(s["msl"].min()) / HPA_TO_PA,
            "min sp / PS (hPa)": float(s["sp"].min()) / HPA_TO_PA,
            "max wind (m/s)": float(np.hypot(s["u10"], s["v10"]).max()),
        }


def ic_stats(f):
    i = np.argmin(np.where(near, f["PS"], np.inf))
    return {
        "min msl (hPa)": np.nan,
        "min sp / PS (hPa)": float(f["PS"][i]),
        "max wind (m/s)": float(np.nanmax(f["wspd_bot"][near])),
        "PS-min offset from best track (km)": float(dist_km[i]),
    }


summary = pd.DataFrame(
    {
        "ERA5 control (0.25°)": source_stats(CTRL_FILES),
        "ERA5 bogus (0.25°)": source_stats(BOGUS_FILES),
        f"E3SM control ({DST_HORZ_GRID})": ic_stats(fld["control"]),
        f"E3SM bogus ({DST_HORZ_GRID})": ic_stats(fld["bogus"]),
    }
).T
summary["best-track p_c (hPa)"] = bt_pc_hpa
summary.round(1)

### Radial profiles

Azimuthal means around the best-track centre in 25 km bins: ERA5 `sp` at 0.25°
and E3SM `PS` on the GLL nodes. Over the ocean, PS is close to MSLP, so the
best-track central pressure is shown at r = 0.

In [ ]:
edges = np.arange(0.0, 800.0 + 25.0, 25.0)
mid = 0.5 * (edges[1:] + edges[:-1])


def radial_mean(r, v):
    idx = np.digitize(r, edges) - 1
    ok = (idx >= 0) & (idx < mid.size)
    return np.array([np.nanmean(v[ok][idx[ok] == k]) if np.any(idx[ok] == k) else np.nan
                     for k in range(mid.size)])


def source_sp(files):
    with xr.open_dataset(files[1]) as s:
        sp = s["sp"].isel(valid_time=0)
        la2, lo2 = np.meshgrid(sp.latitude.values, sp.longitude.values, indexing="ij")
        sel = (np.abs(la2 - bt_center[0]) <= 8) & (np.abs(lo2 - bt_center[1]) <= 8)
        r = haversine_dist(bt_center[0], bt_center[1], la2[sel], lo2[sel])
        return r, sp.values[sel] / HPA_TO_PA


fig, ax = plt.subplots(figsize=(8, 5))
for files, label, style in ((CTRL_FILES, "ERA5 control", "C0--"), (BOGUS_FILES, "ERA5 bogus", "C1--")):
    ax.plot(mid, radial_mean(*source_sp(files)), style, label=label)
for run, style in (("control", "C0o-"), ("bogus", "C1^-")):
    ax.plot(mid, radial_mean(dist_km[near], fld[run]["PS"][near]), style, ms=4,
            label=f"E3SM {run} ({DST_HORZ_GRID})")
ax.plot([0], [bt_pc_hpa], "k*", ms=14, label="best-track p_c")
ax.set_xlabel("distance from best-track centre (km)")
ax.set_ylabel("surface pressure (hPa)")
ax.grid(alpha=0.3)
ax.legend()
ax.set_title(f"Mawar {START_TIME:%Y-%m-%d %HZ}: surface pressure before and after HICCUP")
plt.tight_layout()
fig.savefig(OUT_DIR / "radial_ps_hiccup_mawar.pdf")
plt.show()